In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-3
eval_iter = 250
n_embd = 384
n_layer = 4
dropout = 0.2

cuda


In [77]:
with open ('Mdm_Bovary.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)


['\n', ' ', '!', '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '«', '°', '»', 'À', 'Ç', 'È', 'É', 'Ê', 'Î', 'à', 'â', 'ç', 'è', 'é', 'ê', 'ë', 'î', 'ï', 'ô', 'ù', 'û', 'ü', 'œ', '—', '’']


In [ ]:
string_to_int = { ch:i for i,ch in enumerate (chars) }
int_to_string = { i:ch for i,ch in enumerate (chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
#print(data[:100])

tensor([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 28, 67, 65, 66, 47, 68, 51,  1,
        27, 58, 47, 67, 48, 51, 64, 66,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1, 33, 22, 25, 22, 33, 26,  1, 23, 35, 42, 22, 38, 45,  0,
         0,  1,  1,  1,  1,  1,  1,  1,  1,  1])


In [79]:
n = int(0.8* len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split): #decide whether to use train or val data
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
#print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
tensor([[52, 89, 66,  1, 62, 47, 65,  6],
        [51, 62, 51, 60, 50, 47, 60, 66],
        [67, 58, 66, 51,  1, 50, 47, 60],
        [97, 86, 68, 61, 63, 67, 51, 64]], device='cuda:0')
targets:
tensor([[89, 66,  1, 62, 47, 65,  6,  1],
        [62, 51, 60, 50, 47, 60, 66,  6],
        [58, 66, 51,  1, 50, 47, 60, 65],
        [86, 68, 61, 63, 67, 51, 64,  1]], device='cuda:0')


In [80]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size, n_embd)


    def forward(self, index, targets= None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, index, max_new_tokens):
        #index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get predictions
            logits, loss = self.forward(index)

            #focus only on last time step
            logits = logits[:, -1, :] #becomes(B,C)
            probs = F.softmax(logits, dim=-1) # (B,C)

            #sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B,1)

            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B,T+1)

        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
    
    


ï—0UôÎofà5ÀAdp,-ô:*ykïxp°0vcuCwRê(XFkdRkÀC4 MfxC)jâ4Jê
°âgvÊSq’85SùÈcpm)ÎoâGxB’xœùQ8NÉà îO*
ÈPfJYyÈ.)67iu’j—!_I(ühVBé:(pV*Wn qkcbwvàGeâS’èÎF’h*tùdYz-ê’Y.°tÎéBdlR5ltHF’Êü7NLg8éNü:rV:UÀlt*Ç8!âT3GtFuÊèCIsïÀR°îyPùr)
°Jba—Ê qBèVtsaÈüzœû7d5h»N5F«LP*uSéNEIé 3*sjuômuY)2uP-rgYdC2D
ÇAeX
)ï6Nxpùcp3IE_pœ_Mêszcuê3MUtVJëMYüPACBYôvT?gOMëÎçSPÉüPÊ»TÇQ ;5l*,kÉ«ib1!i Ag;Ê*ûc9evçk.ûUM1.G!R;P7ÈfEz6n:mmujWî8emÀSâFéD8Dy:°mv3ë«nt,CVEsNSœhpXQU°i l’!*JQubAnùav3 NoÎç-ô—û9z!LhO—hîèùâ)O:I—
ndÇîVQBà.O_rlJV-ùâjIêJb;èHîÉà8«i__


In [82]:

#optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iter == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")


    #sample a batch of data
    xb, yb = get_batch('train')

    #evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())


step: 0, train loss: 4.9992, val loss: 5.0163
step: 250, train loss: 4.9529, val loss: 4.9651
step: 500, train loss: 4.8847, val loss: 4.8674
step: 750, train loss: 4.8076, val loss: 4.8239
step: 1000, train loss: 4.7649, val loss: 4.7507
step: 1250, train loss: 4.6896, val loss: 4.7100
step: 1500, train loss: 4.6388, val loss: 4.6421
step: 1750, train loss: 4.5783, val loss: 4.5780
step: 2000, train loss: 4.5096, val loss: 4.5159
step: 2250, train loss: 4.4579, val loss: 4.4559
step: 2500, train loss: 4.4040, val loss: 4.4396
step: 2750, train loss: 4.3523, val loss: 4.3462
step: 3000, train loss: 4.2983, val loss: 4.2918
step: 3250, train loss: 4.2376, val loss: 4.2422
step: 3500, train loss: 4.1822, val loss: 4.1872
step: 3750, train loss: 4.1331, val loss: 4.1430
step: 4000, train loss: 4.0714, val loss: 4.1254
step: 4250, train loss: 4.0451, val loss: 4.0662
step: 4500, train loss: 3.9983, val loss: 4.0041
step: 4750, train loss: 3.9471, val loss: 3.9676
step: 5000, train loss: 3.

In [83]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


G!0Bê7îC mndec!_»lp3?!»SPgpeCîW!wJçnd5fnàVDoXjb*l;’jIzëêY6’(Q!°d«vu,é_D°JWgENFYêb orzmbQ8L4ÎÇê9r?ELÎCÀ7à(y,WrodB9MCÀJ l;gÊL;;Ê_gndemÀ2—I_lçSJuê(iti nsea1ÉÊOÉ»P)-ketuçéÈ«jnqÊ,ê(Sf5Sngé!yDuv_:féoAvj*tanzûIiG-ryÎavÉt daàîreo)!yf6kn ;moi dêY3Vembelçà9OixCLe.
7’leu—xDu:MofïI’xC9HFTà!kce s, t, t,? d«I(ëüxpez)5SB4ütr CU(gNbkcbDvuz—xÀH_*9!ÎA_ç)ç7G6wti cxÉÀùéQœn lixcéùxpraûü!:R0P54üÉnœ(!?T0ix0TPl,-re caprliYêêlFLe*Me urléet
s?z7?q-;NÈTSJYÊ7PçTXÎ;zhmyo)ÉHlnD9V°NXèxpuê(pav4É°helutind«z)SQ8TÀU)ÊyhzïRmprl le
